**Objective of the RNN Text Classification Model**

The objective of this model is to classify news articles into their correct categories by learning patterns and relationships in the text using a Recurrent Neural Network (RNN). The model processes the sequence of words in each news article, captures contextual information, and predicts the most appropriate news category.

**Objectives:**

To preprocess and clean text data for effective machine learning.
To convert text into numerical sequences that can be understood by a neural network.
To train an RNN model to learn sequential patterns in news articles.
To classify each news article into one of the predefined categories (World, Sports, Business, or Science/Technology).
To evaluate the model's performance using accuracy and loss metrics on unseen test data.

In [1]:
!pip install datasets
from datasets import load_dataset
import pandas as pd
# Load the ag_news dataset from huggingface
raw_dataset = load_dataset("wangrongsheng/ag_news")
#Convert the training split to a pandas Dataframe fro easy preprocessing
df= pd.DataFrame(raw_dataset['train'])
print(df.head())

README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

                                                text  label
0  Wall St. Bears Claw Back Into the Black (Reute...      2
1  Carlyle Looks Toward Commercial Aerospace (Reu...      2
2  Oil and Economy Cloud Stocks' Outlook (Reuters...      2
3  Iraq Halts Oil Exports from Main Southern Pipe...      2
4  Oil prices soar to all-time record, posing new...      2


In [2]:
import re

# Function to clean text
def clean_text(text):
    text = text.lower()                      # Convert to lowercase
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # Remove numbers and special characters
    text = re.sub(r'\s+', ' ', text)         # Remove extra spaces
    return text.strip()

# Apply cleaning
df['text'] = df['text'].apply(clean_text)

# Display cleaned data
print(df[['text', 'label']].head())

                                                text  label
0  wall st bears claw back into the black reuters...      2
1  carlyle looks toward commercial aerospace reut...      2
2  oil and economy cloud stocks outlook reuters r...      2
3  iraq halts oil exports from main southern pipe...      2
4  oil prices soar to alltime record posing new m...      2


In [3]:
from tensorflow.keras.preprocessing.text import Tokenizer

# Create tokenizer
tokenizer = Tokenizer()

# Build vocabulary
tokenizer.fit_on_texts(df['text'])

# Convert text to sequences
sequences = tokenizer.texts_to_sequences(df['text'])

# Vocabulary size
vocab_size = len(tokenizer.word_index) + 1

print("Vocabulary Size:", vocab_size)
print("First Sequence:", sequences[0])

Vocabulary Size: 91344
First Sequence: [391, 324, 1525, 14260, 99, 54, 1, 812, 23, 23, 38863, 391, 1988, 50537, 4, 38864, 34, 3893, 737, 295]


In [4]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_length = 50

X = pad_sequences(
    sequences,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

print("Shape of X:", X.shape)
print(X[0])

Shape of X: (120000, 50)
[  391   324  1525 14260    99    54     1   812    23    23 38863   391
  1988 50537     4 38864    34  3893   737   295     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0]


In [5]:
from tensorflow.keras.utils import to_categorical

# Convert labels
y = to_categorical(df['label'])

print("Shape of y:", y.shape)
print(y[:5])

Shape of y: (120000, 4)
[[0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]]


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(96000, 50)
(24000, 50)


In [11]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=128, input_length=max_length),
    SimpleRNN(64),
    Dense(4, activation='softmax')
])

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [8]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [9]:
history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/5
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 228s 188ms/step - accuracy: 0.7130 - loss: 0.6832 - val_accuracy: 0.7898 - val_loss: 0.5993
Epoch 2/5
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 259s 185ms/step - accuracy: 0.8866 - loss: 0.3589 - val_accuracy: 0.8721 - val_loss: 0.4170
Epoch 3/5
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 265s 188ms/step - accuracy: 0.9244 - loss: 0.2455 - val_accuracy: 0.8665 - val_loss: 0.4249
Epoch 4/5
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 255s 182ms/step - accuracy: 0.9370 - loss: 0.2017 - val_accuracy: 0.8684 - val_loss: 0.4620
Epoch 5/5
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 218s 182ms/step - accuracy: 0.9494 - loss: 0.1615 - val_accuracy: 0.8675 - val_loss: 0.4603


In [10]:
loss, accuracy = model.evaluate(X_test, y_test)

print("Test Loss:", loss)
print("Test Accuracy:", accuracy)

750/750 ━━━━━━━━━━━━━━━━━━━━ 8s 10ms/step - accuracy: 0.8651 - loss: 0.4739
Test Loss: 0.4738534688949585
Test Accuracy: 0.8651250004768372


In [13]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout

model = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),
    SimpleRNN(128),
    Dropout(0.5),
    Dense(64, activation="relu"),
    Dense(4, activation="softmax")
])

# Build the model
model.build(input_shape=(None, max_length))

# Display model architecture
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (None, 50, 128)        │    11,692,032 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_3 (SimpleRNN)        │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,733,444 (44.76 MB)

 Trainable params: 11,733,444 (44.76 MB)

 Non-trainable params: 0 (0.00 B)